# Day 4 — ILT 1: Ingestion Patterns Recap — Across GlobalMart's Sources
### GlobalMart Data Engineering · 12:00 PM – 1:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- State GlobalMart's two real ingestion pathways and which tables travel through each
- Explain why Postgres (Supabase) uses CDC via Lakeflow Connect, not a nightly full read
- Explain why the four ADLS file-drop entities use Auto Loader, not a plain `spark.read` loop
- Explain why the REST API and Neo4j graph patterns from Day 3 are **not** part of this production pipeline
- Describe how both pathways land in Bronze with the same audit-column contract
- Trace the path from raw source all the way to `fact_sales`

---

## Agenda

| Time | Topic |
|------|-------|
| 12:00 | GlobalMart architecture recap — 2 pathways, 6 tables |
| 12:10 | Pathway 1 — Postgres (Supabase) via Lakeflow Connect CDC |
| 12:25 | Pathway 2 — ADLS file drops via Auto Loader |
| 12:40 | Why not APIs / Neo4j? (Day 3 side-explorations, not in this pipeline) |
| 12:47 | Cross-pathway comparison + Bronze landing map |
| 12:55 | Q&A |

> **Note on "4 sources":** you'll sometimes hear this session called "all 4 sources." That name is a holdover from an earlier design that had 4 separate *systems* (Postgres, a REST API, Neo4j, ADLS). The confirmed architecture has only **two real systems** — Postgres and ADLS. The "4" that survives is accurate at the *file* level: 4 flat files drop into ADLS (`products`, `customers`, `address`, `payments`). Postgres contributes 2 more tables (`orders`, `order_items`) via CDC. Six tables, two pathways — that's what we're recapping today.

---
## GlobalMart Architecture — Quick Recap

```
┌───────────────────────────────────────────────────────────────────────┐
│                       GLOBALMART'S TWO REAL SOURCES                     │
│                                                                         │
│   ┌─────────────────────────┐          ┌───────────────────────────┐  │
│   │  Postgres (Supabase)    │          │  ADLS File Drops           │  │
│   │  orders, order_items    │          │  products, customers,      │  │
│   │                         │          │  address, payments         │  │
│   └────────────┬────────────┘          └──────────────┬──────────────┘  │
│                │  Lakeflow Connect                     │  Auto Loader   │
│                │  (CDC, WAL-based)                     │  (cloudFiles)  │
│                ▼                                       ▼               │
└────────────────┼───────────────────────────────────────┼───────────────┘
                  │                                       │
                  ▼                                       ▼
┌───────────────────────────────────────────────────────────────────────┐
│                       BRONZE LAYER (Delta Lake)                        │
│   bronze/supabase/orders        bronze/adls/products                   │
│   bronze/supabase/order_items   bronze/adls/customers                  │
│                                  bronze/adls/address                    │
│                                  bronze/adls/payments                   │
└───────────────────────────────────────────────────────────────────────┘
                  │
                  ▼
           SILVER → GOLD (star schema) → fact_sales → Genie
```

### Why Only Two Real Pathways?

GlobalMart's actual production footprint is simple on purpose: an OLTP database for transactional data, and flat files for everything suppliers/ops teams hand off in bulk.

| Pathway | Tables | Why This Pathway |
|---------|--------|-------------------|
| **Postgres CDC** (Lakeflow Connect) | `orders`, `order_items` | High-change-rate transactional rows — need inserts, updates, **and deletes** captured continuously |
| **ADLS Autoloader** | `products`, `customers`, `address`, `payments` | Reference/dimension-shaped data that arrives as periodic file drops — need exactly-once file processing with schema evolution |

You met both mechanisms already: Day 2 (`Day2_ILT1_CDC_Concepts_WAL_Supabase`, `Day2_HOL2_CDC_PostgreSQL_WAL`) walked the Postgres WAL/CDC mechanics by hand, and Day 3 (`Day3_ILT3_Structured_Streaming_AutoLoader`, `Day3_HOL2_ADLS_AutoLoader_Bronze_Customers_Payments`) built Auto Loader Bronze tables for `customers` and `payments`. Today we recap both, add the two remaining ADLS entities (`products`, `address`) and the Postgres CDC entities (`orders`, `order_items`) into one consistent picture, and set up the design principles for the rest of Bronze (next session).

---
## Pathway 1 — Postgres (Supabase) via Lakeflow Connect CDC

### What Is It?
Supabase is GlobalMart's hosted PostgreSQL database — the transactional core. Every order and every order line item is written here first.

### The Ingestion Challenge
```
Problem: Postgres is an OLTP database — optimised for fast writes, not bulk reads.
         Querying the full orders table (millions of rows) every hour kills performance.
         We need CHANGES ONLY — what was inserted/updated/deleted since last run.
         A plain "SELECT * WHERE updated_at > last_run" MISSES deletes entirely.
```

### The Mechanism — WAL-Based CDC

```
Postgres WAL (Write-Ahead Log)
      |
      ▼  logical replication slot — a bookmark that remembers what's been read
Lakeflow Connect (managed, production)
      |
      ▼  INSERT / UPDATE / DELETE events, exactly once
bronze/supabase/orders , bronze/supabase/order_items
      |
      ▼  MERGE (upsert) in Silver
silver.orders_clean , silver.order_items_clean
```

**Key concepts (recap from Day 2):**
- **WAL** = append-only log of every database change, written before the table itself changes
- **Logical replication slot** = a named bookmark in the WAL — so a reader never misses or duplicates events
- **Lakeflow Connect** is the managed pipeline GlobalMart uses in production — it reads the slot and lands CDC events straight into Bronze Delta tables. It's what Day 2 HOL 2's manual `pg_logical_slot_get_changes()` walkthrough was demystifying — same mechanism, no black box in between
- Each row lands with `_cdc_op` (INSERT/UPDATE/DELETE) and audit columns — never resolved to "current state" in Bronze; that's Silver's job via MERGE

**Trigger mode:** continuous (Lakeflow Connect) or `availableNow` scheduled batch, depending on pipeline configuration.

### Why Not Just a Nightly JDBC Pull?

| | CDC (WAL / Lakeflow Connect) | Plain JDBC + watermark |
|--|-----------|------------|
| Captures deletes? | Yes | No — a deleted row leaves no trace |
| Source load | Low (reads WAL, not the table) | Medium (queries the live table) |
| Latency | Near real-time | Depends on schedule |
| What GlobalMart uses in production | ✅ Preferred | Fallback only — useful when a table has no `updated_at` column to watermark on |

> Today's Bronze-build HOL uses a JDBC + watermark script to *simulate* the CDC pathway for `orders`/`order_items` in a notebook — this is what Lakeflow Connect automates for you in production, and it's the same mental model you built by hand in Day 2 HOL 2.

---
## Pathway 2 — ADLS File Drops via Auto Loader

### What Is It?
Four flat files land in the `raw-data/` folder of GlobalMart's shared ADLS Gen2 external location: `products.csv`, `customers.csv`, `addresses.csv`, `payments.csv`. These aren't transactional events — they're periodic drops of reference/dimension-shaped data (product catalog updates, customer master data, address book, payment records).

> **Real path:** `abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data/<entity>/` — registered as the Unity Catalog external location `gbmart-ext-loc`. Landing tables are Unity Catalog managed tables: `gbmart.bronze.customers`, `gbmart.bronze.products`, `gbmart.bronze.addresses`, `gbmart.bronze.payments`.

### The Ingestion Challenge
```
Problem: Files land at unpredictable times.
         Must process each file EXACTLY ONCE — no re-reads, no duplicate rows.
         Schema can evolve (a supplier adds a new column to products.csv).
         Volume varies wildly file to file.
```

### Pattern — Auto Loader (`cloudFiles`)

```python
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",              "csv")
    .option("cloudFiles.schemaLocation",      schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",      "true")
    .option("inferColumnTypes", "true")
    .load(f"{EXTERNAL_LOCATION}/products/")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

query = (
    raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema",        "true")
    .trigger(availableNow=True)
    .toTable("gbmart.bronze.products")
)
```

### What Makes Auto Loader Special (recap from Day 3)

```
Without Auto Loader (spark.read loop):
  Run 1: reads products.csv → 12,000 rows
  Run 2: reads products.csv AGAIN → 12,000 DUPLICATE rows
  Problem: every run re-processes every file

With Auto Loader (cloudFiles):
  Run 1: reads products.csv → 12,000 rows → checkpoint records products.csv
  Run 2: sees products.csv in checkpoint → SKIP
          sees a new file → processes only the new rows
  Result: zero duplicates, exactly-once delivery
```

**Trigger modes:**
- `availableNow=True` → process all pending files, then stop (scheduled batch style) — what we use for GlobalMart's periodic drops
- `processingTime="30 seconds"` → continuous, detect files as they land

> Day 3 HOL 2 already built this exact pattern for `customers` and `payments`. Today's Bronze HOL extends it to all four ADLS entities: `products`, `customers`, `addresses`, `payments`.

---
## Why Not APIs or Neo4j? (Day 3 Side-Explorations)

If you've heard GlobalMart described elsewhere as having "4 source *systems*" — Postgres, a REST API, Neo4j, and ADLS — that was the earlier design. The confirmed architecture (`globalmart_problem_statement_architecture.html`) is explicit:

> *"Two real sources, the way GlobalMart actually has them today. A REST API and a graph database are explored separately on Day 3 as patterns you'll meet in other projects — they don't feed this pipeline."*

### What Day 3 ILT 1 Covered (and why it's still valuable)

| Pattern | What You Learned | Why It's Not in *This* Pipeline |
|---------|-------------------|----------------------------------|
| **REST API ingestion** | Pagination, rate limiting, scheduled HTTP pulls, `mergeSchema` for API version drift | GlobalMart has no enrichment API in scope for `fact_sales` — but you'll meet this pattern on real client projects |
| **Neo4j / Cypher basics** | Nodes, edges, Cypher queries, why graph traversal beats recursive SQL for relationship questions | GlobalMart's supplier/product relationships live in flat reference tables, not a graph DB — again, a pattern for your toolkit, not this build |

**The takeaway:** knowing *more* ingestion patterns than your current project needs is normal and valuable — you just need to correctly identify, for a given project, which 1–2 pathways are actually in play. For GlobalMart, that's CDC + Autoloader. Nothing else touches Bronze.

---
## Cross-Pathway Comparison

| Dimension | Postgres CDC (Lakeflow Connect) | ADLS Autoloader |
|-----------|----------------------------------|-------------------|
| **Tables** | `orders`, `order_items` | `products`, `customers`, `addresses`, `payments` |
| **Data shape** | Transactional event rows | Periodic reference-data file drops |
| **Mechanism** | WAL / logical replication slot | `cloudFiles` streaming source |
| **Exactly-once via** | Replication slot offset | Checkpoint (tracks processed files) |
| **Captures deletes?** | Yes — DELETE is a first-class event | N/A — files are full snapshots, not deltas |
| **Trigger mode** | Continuous / `availableNow` | `availableNow` (batch) or `processingTime` (continuous) |
| **Schema evolution** | Managed by Postgres DDL + Lakeflow | `cloudFiles.schemaEvolutionMode = addNewColumns` |
| **Bronze write mode** | Append (every CDC event is a new row) | Append (every new file's rows are new) |
| **Audit signature** | `_cdc_op`, `_ingested_at` | `_source_file`, `_ingested_at` |

---

## Bronze Landing Map — All 6 Tables

Every table below lives in the **`gbmart` catalog**, `bronze` schema — Unity Catalog managed tables, no manual path bookkeeping needed once the external location and connection are set up:

```
gbmart.bronze.orders          ← CDC events (append) — Lakeflow Connect
gbmart.bronze.order_items     ← CDC events (append) — Lakeflow Connect

gbmart.bronze.products        ← Auto Loader (append)
gbmart.bronze.customers       ← Auto Loader (append)
gbmart.bronze.addresses       ← Auto Loader (append)
gbmart.bronze.payments        ← Auto Loader (append)
```

**Bronze rules — same for both pathways:**
1. Never transform — land raw data as close to source shape as possible
2. Always add audit columns: `_ingested_at`, plus pathway-specific columns (`_source_file` for Autoloader, `_cdc_op` for CDC)
3. Never delete — Bronze is append-only; a source DELETE lands as a DELETE *event*, not a row removal
4. All 6 tables are Delta format, regardless of pathway

---
## What Comes Next

```
Bronze (raw, 6 tables)              Silver (clean, conformed)        Gold (business-ready)
────────────────────────────        ────────────────────────         ────────────────────────
gbmart.bronze.orders           →    gbmart.silver.orders        →    fact_sales
gbmart.bronze.order_items      →    gbmart.silver.order_items   →      (grain: one row per
gbmart.bronze.products         →    gbmart.silver.products           order line item)
gbmart.bronze.customers        →    gbmart.silver.customers     →    dim_customer, dim_product,
gbmart.bronze.addresses        →    gbmart.silver.address            dim_date, dim_address,
gbmart.bronze.payments         →    gbmart.silver.payments           dim_payment_method
```

**Silver transformations (later sessions):**
- Cast data types explicitly (string dates → timestamp)
- Apply MERGE for CDC events (upsert to latest state per `order_id`)
- Deduplicate ADLS file drops (a re-uploaded file shouldn't create duplicate customers)
- Enrich and standardise column names
- Apply data quality rules before anything reaches Gold

This session's job was to make sure the *shape* of Bronze is crystal clear before we formalize its design rules in the next ILT and build it hands-on right after.

---
## Key Takeaways

1. **GlobalMart has two real ingestion pathways, not four** — Postgres CDC and ADLS Autoloader
2. **CDC > watermark JDBC** for transactional sources — captures deletes, lower source load, and is what Lakeflow Connect automates in production
3. **Auto Loader > `spark.read` loop** for files — exactly-once via checkpoint, built-in schema evolution
4. **APIs and Neo4j are real patterns you now know** — just not ones this particular pipeline uses; Day 3 ILT 1 was deliberately a side-exploration
5. **Both pathways land in Bronze as Delta, append-only, with audit columns** — same guarantees regardless of source
6. **Six tables, one Bronze layer** — `orders`, `order_items` (CDC) + `products`, `customers`, `address`, `payments` (Autoloader) — all feeding toward `fact_sales`

---

## Discussion Questions

1. *Why can't Auto Loader be used for the Postgres `orders` table?*

2. *A new ADLS file lands every 5 minutes. Should ingestion use `availableNow=True` or `processingTime`? Why?*

3. *Why does Bronze use append mode for CDC events rather than upsert (MERGE) directly?*

4. *`products.csv` gets a new column `discount_eligible` in next week's drop. What happens to the Bronze write? What config makes this safe?*

5. *If GlobalMart later added a REST weather API for delivery-delay analysis, which pathway from today would it resemble more — CDC or Autoloader? Why?*